#  Deep Dive Feature Exploration
 
 Goals:
 1. Recursively scan all columns in the flattened dataframe.
 2. Extract every potential signal (high variance, non-null).
 3. Correlate with `fraud_flag` to find hidden gems.
 4. Generate a report for `knowledge_base.md`.

In [2]:
import polars as pl
import pandas as pd
import numpy as np

DATA_PATH = "../artifacts/raw_insertions.parquet"

print(f"Loading {DATA_PATH}...")
df = pl.read_parquet(DATA_PATH)

Loading ../artifacts/raw_insertions.parquet...


## 1. Feature Enumeration & Basic filtering

In [3]:
# Derive Target
df = df.with_columns(
    pl.col("fraud_flag").is_not_null().alias("is_fraud").cast(pl.Int8)
)

all_columns = df.columns
total_rows = df.height
print(f"Total Columns: {len(all_columns)}")
print(f"Total Rows: {total_rows}")

# Filter out ID columns for feature analysis, but keep hashes if they might be effectively categorical
# We skip columns that are purely unique IDs (count distinct == count)
# But hashes like 'city_hash' are useful.

potential_features = []
stats = []

# Sampling for speed if needed
if total_rows > 100000:
    df_sample = df.sample(100000, seed=42)
else:
    df_sample = df

print("Scanning columns...")

for col in all_columns:
    if col in ["is_fraud", "fraud_flag", "listing_id"]:
        continue
        
    # Check Type
    dtype = df_sample.schema[col]
    
    # 1. Null Check
    null_count = df_sample[col].null_count()
    coverage = 1.0 - (null_count / df_sample.height)
    
    if coverage < 0.05: # Ignore if < 5% coverage
        continue
        
    # 2. Cardinality
    n_unique = df_sample[col].n_unique()
    
    # Ignore unique IDs (cardinality ~ rows) unless it's numeric range?
    if n_unique > df_sample.height * 0.95 and n_unique > 1000:
        continue
        
    # 3. Variance (if numeric)
    variance = 0.0
    is_numeric = dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    
    if is_numeric:
        variance = df_sample[col].var()
        if variance == 0 or np.isnan(variance):
            continue
    
    # 4. Value Counts (if categorical)
    if not is_numeric and n_unique == 1:
        continue
        
    # If passed filters, add to list
    stats.append({
        "column": col,
        "dtype": str(dtype),
        "coverage": coverage,
        "n_unique": n_unique,
        "is_numeric": is_numeric
    })

print(f"Candidate Features found: {len(stats)}")
df_stats = pl.DataFrame(stats).sort("coverage", descending=True)
print(df_stats.head(20))

Total Columns: 293
Total Rows: 234458
Scanning columns...
Candidate Features found: 109
shape: (20, 5)
┌─────────────────────────────────┬──────────────────────────┬──────────┬──────────┬────────────┐
│ column                          ┆ dtype                    ┆ coverage ┆ n_unique ┆ is_numeric │
│ ---                             ┆ ---                      ┆ ---      ┆ ---      ┆ ---        │
│ str                             ┆ str                      ┆ f64      ┆ i64      ┆ bool       │
╞═════════════════════════════════╪══════════════════════════╪══════════╪══════════╪════════════╡
│ account_created_at              ┆ Datetime(time_unit='us', ┆ 1.0      ┆ 63976    ┆ false      │
│                                 ┆ time_…                   ┆          ┆          ┆            │
│ listing.address.address_hash    ┆ String                   ┆ 1.0      ┆ 68310    ┆ false      │
│ listing.address.country_hash    ┆ String                   ┆ 1.0      ┆ 73       ┆ false      │
│ listing.categ

## 2. Correlation Analysis (Top candidates)

In [4]:
# Convert to pandas for easier correlation
valid_cols = [r["column"] for r in stats]
# Limit to top 50 by coverage/variance for display?
# Or just numeric ones for correlation matrix
numeric_candidates = [r["column"] for r in stats if r["is_numeric"]]
categorical_candidates = [r["column"] for r in stats if not r["is_numeric"]]

print(f"Numeric Candidates: {len(numeric_candidates)}")

# Calculate Correlation with Target
correlations = []
y = df_sample["is_fraud"].to_numpy()

for col in numeric_candidates:
    # Handle nulls for correlation (fill 0)
    x = df_sample[col].fill_null(0).to_numpy()
    
    # Simple correlation
    if x.std() == 0: continue
    
    corr = np.corrcoef(x, y)[0, 1]
    if not np.isnan(corr):
        correlations.append({"feature": col, "correlation": corr, "abs_corr": abs(corr)})

df_corr = pd.DataFrame(correlations).sort_values("abs_corr", ascending=False)
print("\nTop Numeric Correlations with Fraud:")
print(df_corr.head(20))

Numeric Candidates: 23

Top Numeric Correlations with Fraud:
                                       feature  correlation  abs_corr
0   auto_approval_criteria.meta.seonFraudScore     0.458979  0.458979
17                  listing.legacy.ppaPersonId     0.174714  0.174714
16                     listing.legacy.personId     0.174714  0.174714
22                                     user_id     0.151112  0.151112
14           listing.characteristics.yearBuilt    -0.137354  0.137354
12       listing.characteristics.numberOfRooms    -0.125509  0.125509
1                          bundle.initialPrice     0.122245  0.122245
4                        bundle.recurringPrice     0.117247  0.117247
11      listing.characteristics.numberOfFloors    -0.102356  0.102356
3                                bundle.period    -0.091450  0.091450
2                       bundle.oldInitialPrice     0.073045  0.073045
8                listing.characteristics.floor     0.071699  0.071699
15   listing.characteristics.

## 3. Categorical Information Signal (Mutual Information heuristic)

In [7]:
# For high cardinality categoricals, correlation isn't great.
# Let's check fraud rate variance across categories.
# Ideally use Mutual Information, but simple "Max Fraud Rate Deviation" is a quick proxy.

cat_results = []
global_mean = y.mean()

for col in categorical_candidates:
    # Skip high cardinality > 1000 for this quick check to avoid noise
    # unless it's a hash we suspect (like city_hash)
    if "hash" not in col and df_sample[col].n_unique() > 100:
        continue
        
    # Group by category, calc fraud rate
    agg = (
        df_sample.group_by(col)
        .agg([
            pl.len().alias("count"),
            pl.col("is_fraud").mean().alias("fraud_rate")
        ])
        .filter(pl.col("count") > 50) # Min support
    )
    
    if agg.height < 2: continue
    
    # Score = Weighted deviation from mean?
    # Or just Max - Min fraud rate
    max_rate = agg["fraud_rate"].max()
    min_rate = agg["fraud_rate"].min()
    weighted_diff = (max_rate - min_rate) * (agg.height / 10) # rough
    
    cat_results.append({
        "feature": col,
        "max_fraud_rate": max_rate,
        "n_categories_screened": agg.height,
        "spread": max_rate - min_rate
    })

df_cat = pd.DataFrame(cat_results).sort_values("spread", ascending=False)
print("\nTop Categorical Features (Fraud Rate Spread):")
print(df_cat.head(20))


Top Categorical Features (Fraud Rate Spread):
                                         feature  max_fraud_rate  \
44     listing.lister.billing.phoneDay.area_hash        0.892857   
49              listing.lister.email.domain_hash        0.866667   
35      listing.lister.billing.email.domain_hash        0.833333   
2              auto_approval_criteria.meta.state        0.767785   
52                listing.lister.phone.area_hash        0.701923   
7                      listing.address.city_hash        0.642857   
45  listing.lister.billing.phoneDay.country_hash        0.641758   
53             listing.lister.phone.country_hash        0.622951   
4                            contact_emails_hash        0.576793   
47   listing.lister.billing.phoneDay.number_hash        0.555556   
34       listing.lister.billing.address.zip_hash        0.515924   
11                      listing.address.zip_hash        0.461538   
31      listing.lister.billing.address.city_hash        0.400000   
6

In [8]:
# Save results to review
# In Jupyter we just display them
print("Top Stats:")
display(df_stats.head(20))
print("\nTop Correlations:")
display(df_corr.head(20))
print("\nTop Categorical:")
display(df_cat.head(20))

Top Stats:


column,dtype,coverage,n_unique,is_numeric
str,str,f64,i64,bool
"""account_created_at""","""Datetime(time_unit='us', time_…",1.0,63976,false
"""listing.address.address_hash""","""String""",1.0,68310,false
"""listing.address.country_hash""","""String""",1.0,73,false
"""listing.categories""","""String""",1.0,89,false
"""listing.contactForm.size""","""String""",1.0,2,false
…,…,…,…,…
"""listing.address.city_hash""","""String""",0.99991,4705,false
"""listing.address.zip_hash""","""String""",0.9999,3692,false
"""listing.valueAddedServices.bun…","""String""",0.9999,5851,false



Top Correlations:


,feature,correlation,abs_corr
0,auto_approval_criteria.meta.seonFraudScore,0.458979,0.458979
17,listing.legacy.ppaPersonId,0.174714,0.174714
16,listing.legacy.personId,0.174714,0.174714
22,user_id,0.151112,0.151112
14,listing.characteristics.yearBuilt,-0.137354,0.137354
12,listing.characteristics.numberOfRooms,-0.125509,0.125509
1,bundle.initialPrice,0.122245,0.122245
4,bundle.recurringPrice,0.117247,0.117247
11,listing.characteristics.numberOfFloors,-0.102356,0.102356
3,bundle.period,-0.091450,0.091450



Top Categorical:


,feature,max_fraud_rate,n_categories_screened,spread
44,listing.lister.billing.phoneDay.area_hash,0.892857,492,0.892857
49,listing.lister.email.domain_hash,0.866667,66,0.866667
35,listing.lister.billing.email.domain_hash,0.833333,64,0.833333
2,auto_approval_criteria.meta.state,0.767785,4,0.751836
52,listing.lister.phone.area_hash,0.701923,196,0.701923
7,listing.address.city_hash,0.642857,358,0.642857
45,listing.lister.billing.phoneDay.country_hash,0.641758,16,0.638222
53,listing.lister.phone.country_hash,0.622951,16,0.585950
4,contact_emails_hash,0.576793,13,0.576793
47,listing.lister.billing.phoneDay.number_hash,0.555556,78,0.555556
